# 02 — Data Processing


## 1. Install and Import


In [1]:
# !pip install transformers torch tqdm -q


In [ ]:
import json
import os
import re
import random
import pandas as pd
from tqdm import tqdm
from transformers import pipeline

os.makedirs('data/processed', exist_ok=True)

random.seed(42)  

print('Ready')


C:\Users\yassin\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ready


In [2]:
checkpoint_path = 'data/processed/political_filter_checkpoint.jsonl'
POLITICAL_THRESHOLD = 0.7  # confidence needed to keep a tweet
BATCH_SIZE = 512  

## 2. Load Raw Tweets

We read the JSONL file we saved in 01.

We stream it line by line — not loading all 190k rows into memory at once.
This is good practice for large files.

In [ ]:
raw_path = 'data/raw/tweets_raw.jsonl'


print('Loading raw tweets...')
rows = []
with open(raw_path, 'r', encoding='utf-8') as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
print(f'Loaded: {len(df):,} rows')
print(df['ideology'].value_counts())


Loading raw tweets...
Loaded: 190,491 rows
ideology
democrat      97901
republican    92590
Name: count, dtype: int64


## Step 1 — Length Filter


In [4]:
MIN_WORDS = 15

before = len(df)
df['word_count'] = df['text'].str.split().str.len()
df = df[df['word_count'] >= MIN_WORDS].copy()
after = len(df)

print(f'Before: {before:,}')
print(f'After:  {after:,}')
print(f'Removed: {before - after:,} short tweets ({(before-after)/before*100:.1f}%)')
print()
print('Remaining distribution:')
print(df['ideology'].value_counts())


Before: 190,491
After:  171,955
Removed: 18,536 short tweets (9.7%)

Remaining distribution:
ideology
democrat      90974
republican    80981
Name: count, dtype: int64


## Step 2 — Clean Text

**What we remove and why:**

- **URLs** (`https://t.co/xxx`)
- **Mentions** (`@SenWarren`)
- **Hashtags** (`#MAGA`, `#Medicare4All`)
- **Extra whitespace** 


In [ ]:
def clean_tweet(text):
    # Remove URLs
    text = re.sub(r'http\S+', '', text)
    
    # Remove mentions (@username)
    text = re.sub(r'@\w+', '', text)

    # Remove hashtag symbol but keep the word
    text = re.sub(r'#(\w+)', r'\1', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    if len(text.split()) < MIN_WORDS:
        return None

    return text


before = len(df)
df['text'] = df['text'].apply(clean_tweet)
df = df[df['text'].notna()].copy() 
after = len(df)

print(f'Before: {before:,}')
print(f'After:  {after:,}')
print(f'Removed: {before - after:,} (became too short after cleaning)')
print()

print('Example cleaned tweet:')
print(df['text'].iloc[0])


Before: 171,955
After:  168,341
Removed: 3,614 (became too short after cleaning)

Example cleaned tweet:
Happy th birthday to the ! The strength, dedication, and skill of our Sailors including those at Portsmouth Naval Shipyard help keep this country safe, secure, and free. Today we recognize and celebrate their incredible service. 246NavyBirthday


## Step 3 — Political DEBATE Filter

**What is this model?**

Political DEBATE is a DeBERTa model trained specifically on political text.
It uses NLI (Natural Language Inference) — you give it:
1. A text (the tweet)
2. A hypothesis (your question about the tweet)

It answers: is this hypothesis true or false given the text?



In [3]:
import torch

print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB" if torch.cuda.is_available() else "")

GPU available: True
GPU name: NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM: 6.0 GB


In [ ]:
print('Loading Political DEBATE model...')
print('First run downloads ~400MB, subsequent runs load from cache')

political_classifier = pipeline(
    'zero-shot-classification',
    model='mlburnham/Political_DEBATE_large_v1.0',
    device=0,  # -1 = CPU, 0 = GPU if available
    torch_dtype=torch.float16
)

print('Model loaded')

# Test it on 2 examples before running on full data
test_texts = [
    'Happy birthday to the Navy! Thank you for your service.',
    'We must raise taxes on the wealthy and invest in public education now.'
]

print()
print('Testing on 2 examples:')
for text in test_texts:
    result = political_classifier(
        text,
        candidate_labels=['political', 'not political'],
        hypothesis_template='The author of this text {}'
    )
    label = result['labels'][0]
    score = result['scores'][0]
    print(f'Text: {text[:60]}...')
    print(f'Result: {label} (confidence: {score:.2f})')
    print()


Loading Political DEBATE model...
First run downloads ~400MB, subsequent runs load from cache


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 394/394 [00:00<00:00, 622.65it/s]


Model loaded

Testing on 2 examples:
Text: Happy birthday to the Navy! Thank you for your service....
Result: political (confidence: 1.00)

Text: We must raise taxes on the wealthy and invest in public educ...
Result: political (confidence: 1.00)



In [ ]:
# Load checkpoint if exists
processed_ids = set()
if os.path.exists(checkpoint_path):
    print('Checkpoint found, loading...')
    with open(checkpoint_path) as f:
        checkpoint_data = [json.loads(l) for l in f]
    processed_ids = {r['original_index'] for r in checkpoint_data}
    print(f'Already processed: {len(processed_ids):,} rows')
else:
    checkpoint_data = []

# Get rows not yet processed
df = df.reset_index(drop=True)
remaining = df[~df.index.isin(processed_ids)]
print(f'Remaining to process: {len(remaining):,} rows')

# Process in batches
texts = remaining['text'].tolist()
ideologies = remaining['ideology'].tolist()
indices = remaining.index.tolist()

with open(checkpoint_path, 'a') as f:
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc='Filtering'):
        batch_texts = texts[i:i+BATCH_SIZE]
        batch_ideologies = ideologies[i:i+BATCH_SIZE]
        batch_indices = indices[i:i+BATCH_SIZE]

        results = political_classifier(
            batch_texts,
            candidate_labels=['political', 'not political'],
            hypothesis_template='The author of this text {}',
            truncation=True,
            max_length=128
        )
        print(f"VRAM: {torch.cuda.memory_reserved(0)/1024**2:.0f}MB")
        for j, result in enumerate(results):
            is_political = (
                result['labels'][0] == 'political' and
                result['scores'][0] >= POLITICAL_THRESHOLD
            )
            row = {
                'original_index': batch_indices[j],
                'text': batch_texts[j],
                'ideology': batch_ideologies[j],
                'is_political': is_political,
                'political_score': result['scores'][0]
            }
            f.write(json.dumps(row) + '\n')
            checkpoint_data.append(row)
        if len(checkpoint_data) >100000:
            break  # stop after 100k to avoid long runs during testing

print('Filtering complete')


Checkpoint found, loading...
Already processed: 100,000 rows
Remaining to process: 68,341 rows


Filtering:   0%|          | 0/134 [03:03<?, ?it/s]

VRAM: 2540MB
Filtering complete


In [8]:
# Load results and apply filter

with open(checkpoint_path) as f:
    checkpoint_data = [json.loads(l) for l in f]


df_filtered = pd.DataFrame(checkpoint_data)
df_political = df_filtered[df_filtered['is_political'] == True].copy()

print(f'Total processed: {len(df_filtered):,}')
print(f'Political:       {len(df_political):,} ({len(df_political)/len(df_filtered)*100:.1f}%)')
print(f'Non-political:   {len(df_filtered) - len(df_political):,}')
print()
print('Political tweets by ideology:')
print(df_political['ideology'].value_counts())


Total processed: 100,512
Political:       83,769 (83.3%)
Non-political:   16,743

Political tweets by ideology:
ideology
democrat      49946
republican    33823
Name: count, dtype: int64


## Step 4 — Topic Detection

**Why detect topics?**

The training format is:
```
[INST] You are a Democratic senator speaking about {topic}.
Express your position. [/INST] {tweet}
```

Without topic detection, you cannot fill `{topic}`.



In [ ]:
TOPICS = [
    'healthcare',
    'immigration',
    'taxes and economy',
    'climate and environment',
    'guns and second amendment',
    'education',
    'military and national security',
    'crime and justice',
    'voting rights and democracy',
    'social equality and civil rights',
]

print('Topics we will detect:')
for t in TOPICS:
    print(f'  - {t}')


Topics we will detect:
  - healthcare
  - immigration
  - taxes and economy
  - climate and environment
  - guns and second amendment
  - education
  - military and national security
  - crime and justice
  - voting rights and democracy
  - social equality and civil rights


In [6]:
# Test topic detection on 3 examples first
test_cases = [
    'We must expand Medicare to cover every American. Healthcare is a right.',
    'The border crisis is out of control. We need to deport illegal aliens now.',
    'Tax cuts for billionaires while workers struggle is morally wrong.'
]

print('Testing topic detection:')
for text in test_cases:
    result = political_classifier(
        text,
        candidate_labels=TOPICS,
        hypothesis_template='This text is about {}'
    )
    print(f'Text:  {text[:70]}...')
    print(f'Topic: {result["labels"][0]} (score: {result["scores"][0]:.2f})')
    print()


Testing topic detection:
Text:  We must expand Medicare to cover every American. Healthcare is a right...
Topic: healthcare (score: 1.00)

Text:  The border crisis is out of control. We need to deport illegal aliens ...
Topic: immigration (score: 1.00)

Text:  Tax cuts for billionaires while workers struggle is morally wrong....
Topic: taxes and economy (score: 1.00)



In [ ]:
topic_checkpoint_path = 'data/processed/topic_checkpoint.jsonl'
TOPIC_THRESHOLD = 0.7
BATCH_SIZE = 512

# Load checkpoint if exists
topic_processed_list = []
topic_processed_ids = set()

if os.path.exists(topic_checkpoint_path):
    print('Topic checkpoint found, loading...')
    with open(topic_checkpoint_path) as f:
        for line in f:
            r = json.loads(line)
            topic_processed_list.append(r)
            topic_processed_ids.add(r['original_index'])
    print(f'Already processed: {len(topic_processed_ids):,} rows')

# Get unprocessed rows
df_political = df_political.reset_index(drop=True)
remaining_topic = df_political[
    ~df_political['original_index'].isin(topic_processed_ids)
]
print(f'Remaining: {len(remaining_topic):,} rows')

texts     = remaining_topic['text'].tolist()
ideologies = remaining_topic['ideology'].tolist()
indices   = remaining_topic['original_index'].tolist()

with open(topic_checkpoint_path, 'a') as f:
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc='Topics'):
        batch_texts      = texts[i:i+BATCH_SIZE]
        batch_ideologies = ideologies[i:i+BATCH_SIZE]
        batch_indices    = indices[i:i+BATCH_SIZE]

        results = political_classifier(
            batch_texts,
            candidate_labels=TOPICS,
            hypothesis_template='This text is about {}',
            truncation=True,      # ← same as what worked
            max_length=128 ,       # ← same as what worked
            batch_size=256
        )

        print(f"VRAM: {torch.cuda.memory_reserved(0)/1024**2:.0f}MB")

        for j, result in enumerate(results):
            top_topic = result['labels'][0]
            top_score = result['scores'][0]
            row = {
                'original_index': batch_indices[j],
                'text':           batch_texts[j],
                'ideology':       batch_ideologies[j],
                'topic':          top_topic if top_score >= TOPIC_THRESHOLD else 'general',
                'topic_score':    top_score
            }
            f.write(json.dumps(row) + '\n')
            topic_processed_list.append(row)
            topic_processed_ids.add(batch_indices[j])

print(f'Topic detection complete — {len(topic_processed_list):,} rows')

Topic checkpoint found, loading...
Already processed: 2,560 rows
Remaining: 81,209 rows


Topics:   1%|          | 1/159 [00:37<1:37:32, 37.04s/it]

VRAM: 4166MB


Topics:   1%|▏         | 2/159 [01:16<1:40:07, 38.27s/it]You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


VRAM: 4166MB


...

Topics:  99%|█████████▉| 158/159 [1:34:41<00:24, 24.56s/it]

VRAM: 5030MB


Topics: 100%|██████████| 159/159 [1:34:56<00:00, 35.82s/it]

VRAM: 5030MB
Topic detection complete — 83,769 rows


: 

## Step 5 — Sample 5k Per Ideology


In [1]:
import pandas as pd

df_topics = pd.read_json(
    "data/processed/topic_checkpoint.jsonl",
    lines=True
)

print(df_topics.shape)

(83769, 5)


In [2]:
topic_counts = df_topics["topic"].value_counts()

print(topic_counts)

topic
general                             26408
military and national security      11256
taxes and economy                    8785
healthcare                           8737
social equality and civil rights     5944
climate and environment              5806
voting rights and democracy          5388
immigration                          3924
education                            3560
crime and justice                    3484
guns and second amendment             477
Name: count, dtype: int64


In [5]:
pd.crosstab(df_topics['ideology'], df_topics['topic'])

topic,climate and environment,crime and justice,education,general,guns and second amendment,healthcare,immigration,military and national security,social equality and civil rights,taxes and economy,voting rights and democracy
ideology,,,,,,,,,,,
democrat,4653,2205,2338,15960,242,6172,1128,4567,5125,3917,3639
republican,1153,1279,1222,10448,235,2565,2796,6689,819,4868,1749


In [7]:
# Remove noise topics
df_clean = df_topics[
    ~df_topics['topic'].isin(['general', 'guns and second amendment'])
].copy()

print(f'After removing noise: {len(df_clean):,} rows')
print(df_clean['ideology'].value_counts())

# Sample 3k per ideology preserving natural topic distribution
dem_pool = df_clean[df_clean['ideology'] == 'democrat']
rep_pool = df_clean[df_clean['ideology'] == 'republican']

dem_sample = dem_pool.sample(5000, random_state=42)
rep_sample = rep_pool.sample(5000, random_state=42)

df_sample = pd.concat([dem_sample, rep_sample]).reset_index(drop=True)



After removing noise: 56,884 rows
ideology
democrat      33744
republican    23140
Name: count, dtype: int64


In [8]:
print(f'\nFinal sample: {len(df_sample):,} rows')
print('Democrat topic distribution in sample:')
dem_sample['topic'].value_counts()



Final sample: 10,000 rows
Democrat topic distribution in sample:


topic
healthcare                          845
social equality and civil rights    749
military and national security      710
climate and environment             704
taxes and economy                   576
voting rights and democracy         562
education                           372
crime and justice                   328
immigration                         154
Name: count, dtype: int64

In [9]:
print('Republican topic distribution in sample:')
rep_sample['topic'].value_counts()

Republican topic distribution in sample:


topic
military and national security      1437
taxes and economy                   1055
immigration                          605
healthcare                           559
voting rights and democracy          344
crime and justice                    287
climate and environment              275
education                            257
social equality and civil rights     181
Name: count, dtype: int64

## Step 6 — Format for Mistral Instruct


Mistral-Instruct was trained on a specific format.
Your fine-tuning data must use the SAME format.
Otherwise there is a mismatch between training and inference.

**The Mistral Instruct template:**
```
<s>[INST] instruction here [/INST] response here</s>
```

- `<s>` = start of sequence token
- `[INST]` = start of instruction
- `[/INST]` = end of instruction, model starts responding here
- `</s>` = end of sequence token



In [11]:
import json

In [12]:
IDEOLOGY_LABELS = {
    'democrat':   'Democratic',
    'republican': 'Republican'
}

def format_for_mistral(row):
    party = IDEOLOGY_LABELS[row['ideology']]
    topic = row['topic']
    text  = row['text']

    instruction = (
        f'You are a {party} senator speaking about {topic}. '
        f'Express your position.'
    )

    formatted = f'<s>[INST] {instruction} [/INST]\n{text}</s>'
    return formatted


df_sample['formatted'] = df_sample.apply(format_for_mistral, axis=1)

# Save
output_path = 'data/processed/train_ready.jsonl'
with open(output_path, 'w', encoding='utf-8') as f:
    for _, row in df_sample.iterrows():
        record = {
            'text':      row['formatted'],
            'ideology':  row['ideology'],
            'topic':     row['topic'],
            'raw_tweet': row['text']
        }
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

print(f'Saved {len(df_sample):,} training examples to {output_path}')

# Verify
print('\nSample Democrat:')
print(df_sample[df_sample['ideology']=='democrat'].iloc[0]['formatted'])
print('\nSample Republican:')
print(df_sample[df_sample['ideology']=='republican'].iloc[0]['formatted'])

Saved 10,000 training examples to data/processed/train_ready.jsonl

Sample Democrat:
<s>[INST] You are a Democratic senator speaking about social equality and civil rights. Express your position. [/INST]
Our AAPI communities make our state and our country stronger. This AAPIHeritageMonth we celebrate the contributions of Asian Americans and Pacific Islanders to our nation and we must also continue to stand up and speak out against anti-Asian sentiment and discrimination</s>

Sample Republican:
<s>[INST] You are a Republican senator speaking about immigration. Express your position. [/INST]
President Biden says he cant guarantee swift action to address our nations border crisis. Thats not good enough for Texans and the American people. SecureTheBorder</s>
